## 1. Setup and Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR   = Path(r"C:\Users\skazempour\Documents\StockTwits\Code")
DATA_DIR   = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\v1\data\csv")

INPUT_FOLDER  = DATA_DIR / "merged_with_crsp_mlcrowd"
OUTPUT_FOLDER = DATA_DIR / "features_mlcrowd"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE   = OUTPUT_FOLDER / "features_05_sentiment_dynamics_cohorts.pkl"

# =============================================================================
# PARAMETERS
# =============================================================================
HORIZONS  = [5, 21, 63, 250]   # trading days (1 wk / 1 mo / 1 qtr / 1 yr)
K_VALUES  = [3, 5, 10]          # conviction run lengths
N_VALUES  = [5, 10, 50]         # first-mover N posts
SHRINK    = 20.0                 # Bayesian pseudo-count for specialist ratio carry

print(f"Input : {INPUT_FOLDER}")
print(f"Output: {OUTPUT_FILE}")

## 2. Load Trading Day Calendar from Fama-French Data

In [ ]:
import pandas_datareader.data as pdr

print("Fetching Fama-French trading days...")
ff = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start="2009-01-01")[0]
trading_days = pd.DatetimeIndex(ff.index).normalize().unique().sort_values()
# Build integer index lookup  {Timestamp -> int}
td_idx = {d: i for i, d in enumerate(trading_days)}
td_sorted = sorted(td_idx, key=lambda d: td_idx[d])

print(f"Trading calendar: {len(trading_days):,} days "
      f"({trading_days[0].date()} to {trading_days[-1].date()})")

## 3. Load Sample Data for Development

In [ ]:
files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith(".csv")])
print(f"Available files: {len(files)}")

sample_file = INPUT_FOLDER / files[-1]
print(f"Loading: {sample_file.name}")
df_sample = pd.read_csv(sample_file)
df_sample["date"]       = pd.to_datetime(df_sample["date"])
df_sample["created_at"] = pd.to_datetime(df_sample["created_at"])

print(f"Shape: {df_sample.shape}")
print(f"Date range: {df_sample['date'].min().date()} to {df_sample['date'].max().date()}")
print(f"Unique symbols: {df_sample['symbol'].nunique():,}  Users: {df_sample['user_id'].nunique():,}")
print(f"Sentiment distribution:\n{df_sample['sentiment'].value_counts(dropna=False)}")
display(df_sample.head(3))

## 4. Helper Functions

In [ ]:
def gini_coeff(arr):
    """Gini concentration coefficient of user post counts (0 = equal, 1 = monopoly)."""
    a = np.sort(np.array(arr, dtype=float))
    n = len(a); s = a.sum()
    if n == 0 or s == 0:
        return 0.0
    return (2 * np.dot(np.arange(1, n + 1), a)) / (n * s) - (n + 1) / n


def get_year_td_slice(df, trading_days):
    """Return trading_days restricted to the date range in df (with 250-day padding left)."""
    data_start = df["date"].min()
    data_end   = df["date"].max()
    # Extend start 250 trading days earlier for rolling windows
    start_idx  = max(0, td_idx.get(pd.Timestamp(data_start.date()), 0) - 250)
    valid = trading_days[
        (trading_days >= td_sorted[start_idx]) & (trading_days <= data_end)
    ]
    return valid

## 5a. Feature Family A ? Sentiment Flips (Features 5-9)

In [ ]:
def calc_flip_features(df, horizons=HORIZONS, prev_daily_dir=None):
    """
    Features 5-9: Bear-to-Bull / Bull-to-Bear flip counts and ratios.

    For each horizon H (trading days): among users posting on (symbol, date) t,
    count those whose most recent PRIOR tagged post on that symbol (within H td)
    had the opposite direction.  No look-ahead: only uses data up to t.

    Parameters
    ----------
    prev_daily_dir : pd.DataFrame or None
        (user_id, symbol, date, direction) rows from the previous year, used so
        Jan-1 users can flip relative to Dec-31 of the prior year.
    """
    tagged = df[df["sentiment"].isin(["Bullish", "Bearish"])].copy()
    tagged["direction"] = tagged["sentiment"].map({"Bullish": 1, "Bearish": -1})

    # Collapse to one direction per (user, symbol, date): majority wins
    daily = (
        tagged.groupby(["user_id", "symbol", "date"])["direction"]
        .mean().apply(lambda x: 1 if x > 0 else -1).reset_index()
        .sort_values(["user_id", "symbol", "date"]).reset_index(drop=True)
    )
    if prev_daily_dir is not None and len(prev_daily_dir) > 0:
        daily = pd.concat([prev_daily_dir, daily]).sort_values(
            ["user_id", "symbol", "date"]).reset_index(drop=True)

    g = daily.groupby(["user_id", "symbol"])
    daily["prev_dir"]  = g["direction"].shift(1)
    daily["prev_date"] = g["date"].shift(1)
    daily["cur_td"]    = daily["date"].map(td_idx)
    daily["prev_td"]   = daily["prev_date"].map(td_idx)
    daily["td_gap"]    = daily["cur_td"] - daily["prev_td"]
    daily["b2b"]  = (daily["prev_dir"] == -1) & (daily["direction"] == 1)
    daily["b2br"] = (daily["prev_dir"] ==  1) & (daily["direction"] == -1)

    # Filter to current year only (drop carry rows used only for gap calc)
    if prev_daily_dir is not None and len(prev_daily_dir) > 0:
        min_cur = df["date"].min()
        daily = daily[daily["date"] >= min_cur]

    n_act = (
        daily.groupby(["symbol", "date"])["user_id"].nunique()
        .rename("n_tagged_users").reset_index()
    )
    result = n_act.copy()
    for H in horizons:
        w = daily[daily["td_gap"].notna() & (daily["td_gap"] <= H)]
        agg = w.groupby(["symbol", "date"]).agg(
            b2b=("b2b", "sum"), b2br=("b2br", "sum")
        ).reset_index()
        agg.columns = ["symbol", "date",
                       f"bear_to_bull_flips_{H}", f"bull_to_bear_flips_{H}"]
        agg[f"net_flip_count_{H}"] = (
            agg[f"bear_to_bull_flips_{H}"] - agg[f"bull_to_bear_flips_{H}"]
        )
        result = result.merge(agg, on=["symbol", "date"], how="left")
        for c in [f"bear_to_bull_flips_{H}", f"bull_to_bear_flips_{H}",
                  f"net_flip_count_{H}"]:
            result[c] = result[c].fillna(0)
        result[f"bear_to_bull_ratio_{H}"] = (
            result[f"bear_to_bull_flips_{H}"] /
            result["n_tagged_users"].replace(0, np.nan)
        )
        result[f"bull_to_bear_ratio_{H}"] = (
            result[f"bull_to_bear_flips_{H}"] /
            result["n_tagged_users"].replace(0, np.nan)
        )

    # Return current year's daily_dir so next year can reference it
    cur_daily = daily[daily["date"] >= df["date"].min()][
        ["user_id", "symbol", "date", "direction"]
    ].copy()
    return result.drop(columns=["n_tagged_users"]), cur_daily

## 5b. Feature Family B ? Conviction Index (Feature 10)

In [ ]:
def calc_conviction_features(df, k_values=K_VALUES, prev_daily_dir=None):
    """
    Feature 10: % of users today with ? K consecutive same-direction tagged posts
    (counting all prior posts on that symbol, including carry from previous years).

    A 'consecutive streak' resets whenever the user's direction changes.
    """
    tagged = df[df["sentiment"].isin(["Bullish", "Bearish"])].copy()
    tagged["direction"] = tagged["sentiment"].map({"Bullish": 1, "Bearish": -1})
    daily = (
        tagged.groupby(["user_id", "symbol", "date"])["direction"]
        .mean().apply(lambda x: 1 if x > 0 else -1).reset_index()
        .sort_values(["user_id", "symbol", "date"]).reset_index(drop=True)
    )
    if prev_daily_dir is not None and len(prev_daily_dir) > 0:
        daily = pd.concat([prev_daily_dir, daily]).sort_values(
            ["user_id", "symbol", "date"]).reset_index(drop=True)

    g = daily.groupby(["user_id", "symbol"])
    daily["prev_dir"]    = g["direction"].shift(1)
    daily["dir_changed"] = (daily["direction"] != daily["prev_dir"]).fillna(True)
    daily["run_id"]      = g["dir_changed"].cumsum()
    daily["streak"]      = (
        daily.groupby(["user_id", "symbol", "run_id"]).cumcount() + 1
    )

    if prev_daily_dir is not None and len(prev_daily_dir) > 0:
        daily = daily[daily["date"] >= df["date"].min()]

    n_act = (
        daily.groupby(["symbol", "date"])["user_id"].nunique()
        .rename("n_active").reset_index()
    )
    result = n_act.copy()
    for K in k_values:
        conv = (
            daily.assign(hc=daily["streak"] >= K)
            .groupby(["symbol", "date"])["hc"]
            .sum().rename(f"n_conv_{K}").reset_index()
        )
        result = result.merge(conv, on=["symbol", "date"], how="left")
        result[f"conviction_index_{K}"] = (
            result[f"n_conv_{K}"] / result["n_active"].replace(0, np.nan)
        )
    return result.drop(
        columns=["n_active"] + [f"n_conv_{K}" for K in k_values]
    )

## 5c. Feature Family C ? First-Mover Sentiment (Feature 16)

In [ ]:
def calc_first_mover_features(df, n_values=N_VALUES):
    """
    Feature 16: Sentiment of the first N users to post about a stock on day t.

    Posts are sorted by created_at within (symbol, date).  Only tagged
    (Bullish/Bearish) posts count toward the N; untagged posts are skipped.
    Output: first_mover_net_sent_N, first_mover_volume_N for each N.
    """
    tagged = df[df["sentiment"].isin(["Bullish", "Bearish"])].copy()
    tagged["direction"] = tagged["sentiment"].map({"Bullish": 1, "Bearish": -1})
    tagged = tagged.sort_values(["symbol", "date", "created_at"])

    result = None
    for N in n_values:
        fn = tagged.groupby(["symbol", "date"]).head(N)
        agg = fn.groupby(["symbol", "date"]).agg(
            bull=("direction", lambda x: (x == 1).sum()),
            bear=("direction", lambda x: (x == -1).sum()),
            vol =("direction", "count"),
        ).reset_index()
        denom = (agg["bull"] + agg["bear"]).replace(0, np.nan)
        agg[f"first_mover_net_sent_{N}"] = (agg["bull"] - agg["bear"]) / denom
        agg[f"first_mover_volume_{N}"]   = agg["vol"]
        keep = agg[["symbol", "date",
                    f"first_mover_net_sent_{N}", f"first_mover_volume_{N}"]]
        result = keep if result is None else result.merge(
            keep, on=["symbol", "date"], how="outer"
        )
    return result

## 5d. Feature Family D ? User Cohort Composition (Features 17-26)

In [ ]:
def calc_cohort_features(df, carry=None, horizons=None):
    """
    Features 17-26: who is talking, and how concentrated is the crowd?

    Requires a carry_state dict that persists across years in the main loop.
    On the first call pass carry=None (initialises automatically).

    carry keys
    ----------
    sym_last   : {(uid, sym) -> last date posted (Timestamp)}
    global_cnt : {uid -> total posts across all prior years}
    sym_set    : {uid -> set of unique symbols posted across all prior years}
    ah_cnt     : {uid -> after-hours post count across all prior years}

    Features produced
    -----------------
    fresh_blood_count_H, fresh_blood_ratio_H  (Features 17-18)  H in horizons
    reentry_volume_H                           (Feature 19)
    whale_dominance, minnow_dominance          (Features 20-21)
    night_owl_ratio, day_trader_ratio          (Features 22-23)
    retention_rate_H                           (Feature 24)
    crowding_gini                              (Feature 25)
    specialist_ratio                           (Feature 26)
    """
    if horizons is None:
        horizons = HORIZONS[:2]  # [5, 21] by default; 63/250 rarely used here
    if carry is None:
        carry = dict(sym_last={}, global_cnt={}, sym_set={}, ah_cnt={})

    df = df.copy()
    df["date"]       = pd.to_datetime(df["date"])
    df["created_at"] = pd.to_datetime(df["created_at"])
    df = df.sort_values(["date", "created_at", "message_id"]).reset_index(drop=True)

    # ?? daily user-symbol table ????????????????????????????????????????????
    daily = (
        df.groupby(["user_id", "symbol", "date"])
        .agg(n=("message_id", "count"), ah=("is_after_hours", "any"))
        .reset_index().sort_values(["user_id", "symbol", "date"])
    )
    g = daily.groupby(["user_id", "symbol"])
    daily["prev_date"] = g["date"].shift(1)
    daily["prev_date"] = daily.apply(
        lambda r: carry["sym_last"].get((r["user_id"], r["symbol"]))
        if pd.isna(r["prev_date"]) else r["prev_date"], axis=1
    )
    daily["gap_days"] = (
        daily["date"] - pd.to_datetime(daily["prev_date"])
    ).dt.days

    # ?? fresh blood / re-entry / retention ????????????????????????????????
    for H in horizons:
        hc = H * 365 / 252   # approximate calendar-day equivalent
        daily[f"fresh_{H}"]   = (
            daily["gap_days"].isna() | (daily["gap_days"] > hc)
        )
        daily[f"reentry_{H}"] = (
            daily["gap_days"].notna() & (daily["gap_days"] > hc) &
            daily.apply(
                lambda r: carry["sym_last"].get(
                    (r["user_id"], r["symbol"])) is not None, axis=1
            )
        )
        daily[f"retain_{H}"]  = (
            daily["gap_days"].notna() & (daily["gap_days"] <= hc)
        )

    # ?? global cumulative post count ???????????????????????????????????????
    df["prior_g"]    = df["user_id"].map(lambda u: carry["global_cnt"].get(u, 0))
    df["yr_cc"]      = df.groupby("user_id").cumcount()
    df["cum_before"] = df["prior_g"] + df["yr_cc"]

    # daily whale threshold = 99th percentile of cum_before across all users that day
    dth = (
        df.groupby("date")["cum_before"].quantile(0.99)
        .rename("p99").reset_index()
    )
    df = df.merge(dth, on="date", how="left")
    df["is_whale"]  = df["cum_before"] >= df["p99"]
    df["is_minnow"] = df["cum_before"] < 5

    # ?? night-owl / day-trader (expanding from carry) ?????????????????????
    df["prior_ah"] = df["user_id"].map(lambda u: carry["ah_cnt"].get(u, 0))
    df["yr_ah_cc"] = (
        df.groupby("user_id")["is_after_hours"].cumsum()
        .shift(1).fillna(0)
    )
    df["cum_ah"] = df["prior_ah"] + df["yr_ah_cc"]
    ah_ratio = df["cum_ah"] / df["cum_before"].replace(0, np.nan)
    df["is_night_owl"]  = ah_ratio > 0.8
    df["is_day_trader"] = ah_ratio < 0.2

    # ?? specialist ratio: unique symbols per user strictly before today ????
    first_occ = (
        df.groupby(["user_id", "symbol"])["date"].min()
        .reset_index().rename(columns={"date": "first_dt"})
    )

    def _count_prior_syms(grp):
        uid = grp.name
        fo_dates = np.sort(
            first_occ[first_occ["user_id"] == uid]["first_dt"]
            .values.astype("int64")
        )
        return grp["date"].apply(
            lambda d: int(np.searchsorted(fo_dates, d.value, side="left"))
        )

    df["yr_nsyms_before"] = df.groupby(
        "user_id", group_keys=False
    ).apply(_count_prior_syms)
    df["prior_nsyms"] = df["user_id"].map(
        lambda u: len(carry["sym_set"].get(u, set()))
    )
    df["total_nsyms"] = df["prior_nsyms"] + df["yr_nsyms_before"]
    df["is_specialist"] = df["total_nsyms"] <= 3

    # ?? crowding (Gini coefficient of user post counts per symbol-day) ????
    gini_s = (
        df.groupby(["symbol", "date"], group_keys=False)
        .apply(lambda g: gini_coeff(g.groupby("user_id").size().values))
        .rename("crowding_gini").reset_index()
    )

    # ?? aggregate to (symbol, date) ???????????????????????????????????????
    sym_a = df.groupby(["symbol", "date"]).agg(
        n_posts    =("message_id", "count"),
        whale_n    =("is_whale",    "sum"),
        minnow_n   =("is_minnow",   "sum"),
        no_n       =("is_night_owl","sum"),
        dt_n       =("is_day_trader","sum"),
        spec_n     =("is_specialist","sum"),
    ).reset_index()

    day_a = daily.groupby(["symbol", "date"]).agg(
        n_u=("user_id", "nunique")
    ).reset_index()
    for H in horizons:
        ha = (
            daily.groupby(["symbol", "date"])[
                [f"fresh_{H}", f"reentry_{H}", f"retain_{H}"]
            ].sum().reset_index()
        )
        day_a = day_a.merge(ha, on=["symbol", "date"], how="left")
        day_a[f"fresh_blood_count_{H}"] = day_a[f"fresh_{H}"]
        day_a[f"fresh_blood_ratio_{H}"] = (
            day_a[f"fresh_{H}"] / day_a["n_u"].replace(0, np.nan)
        )
        day_a[f"reentry_volume_{H}"]    = day_a[f"reentry_{H}"]
        day_a[f"retention_rate_{H}"]    = (
            day_a[f"retain_{H}"] / day_a["n_u"].replace(0, np.nan)
        )
        day_a.drop(
            columns=[f"fresh_{H}", f"reentry_{H}", f"retain_{H}"],
            inplace=True
        )

    result = sym_a.merge(
        day_a.drop(columns=["n_u"]), on=["symbol", "date"], how="left"
    )
    result = result.merge(gini_s, on=["symbol", "date"], how="left")

    n = result["n_posts"].replace(0, np.nan)
    result["whale_dominance"]  = result["whale_n"]  / n
    result["minnow_dominance"] = result["minnow_n"] / n
    result["night_owl_ratio"]  = result["no_n"]     / n
    result["day_trader_ratio"] = result["dt_n"]     / n
    result["specialist_ratio"] = result["spec_n"]   / n
    result.drop(
        columns=["whale_n", "minnow_n", "no_n", "dt_n", "spec_n"],
        inplace=True
    )

    # ?? update carry state (called once per year after this function) ?????
    new_carry = {
        "sym_last"   : dict(carry["sym_last"]),
        "global_cnt" : dict(carry["global_cnt"]),
        "sym_set"    : {k: set(v) for k, v in carry["sym_set"].items()},
        "ah_cnt"     : dict(carry["ah_cnt"]),
    }
    for _, row in daily.iterrows():
        new_carry["sym_last"][(row["user_id"], row["symbol"])] = row["date"]
    for uid in df["user_id"].unique():
        u_df = df[df["user_id"] == uid]
        new_carry["global_cnt"][uid] = (
            new_carry["global_cnt"].get(uid, 0) + len(u_df)
        )
        new_carry["ah_cnt"][uid] = (
            new_carry["ah_cnt"].get(uid, 0) +
            int(u_df["is_after_hours"].sum())
        )
        new_carry["sym_set"][uid] = (
            new_carry["sym_set"].get(uid, set()) |
            set(u_df["symbol"].unique())
        )
    return result, new_carry

## 6. Test on Sample Data

In [ ]:
print("Testing flip features...")
flip_sample, _ = calc_flip_features(df_sample, horizons=[5, 21])
print(f"  Flip features shape: {flip_sample.shape}")
print(f"  Columns: {[c for c in flip_sample.columns if c not in ('symbol','date')][:6]}...")

print("\nTesting conviction features...")
conv_sample = calc_conviction_features(df_sample, k_values=[3, 5])
print(f"  Conviction features shape: {conv_sample.shape}")

print("\nTesting first-mover features...")
fm_sample = calc_first_mover_features(df_sample, n_values=[5, 10])
print(f"  First-mover features shape: {fm_sample.shape}")

print("\nTesting cohort features...")
cohort_sample, carry_sample = calc_cohort_features(
    df_sample, horizons=[21, 63]
)
print(f"  Cohort features shape: {cohort_sample.shape}")

# Merge all
all_sample = (
    flip_sample
    .merge(conv_sample,   on=["symbol", "date"], how="outer")
    .merge(fm_sample,     on=["symbol", "date"], how="outer")
    .merge(cohort_sample, on=["symbol", "date"], how="outer")
    .sort_values(["symbol", "date"]).reset_index(drop=True)
)
print(f"\nMerged sample features: {all_sample.shape}")
print(f"Feature columns ({len(all_sample.columns)-2}): {sorted([c for c in all_sample.columns if c not in ('symbol','date')])[:10]}...")
display(all_sample.head(10))

## 7. Visualize & Validate Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Net flip distribution (H=21)
col = "net_flip_count_21"
if col in all_sample:
    axes[0,0].hist(all_sample[col].dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[0,0].set_title("Net Flip Count (H=21)"); axes[0,0].set_xlabel(col)

# 2. Conviction index K=3
col2 = "conviction_index_3"
if col2 in all_sample:
    vals = all_sample[col2].dropna()
    axes[0,1].hist(vals, bins=30, edgecolor="black", alpha=0.7, color="orange")
    axes[0,1].set_title(f"Conviction Index K=3 (mean={vals.mean():.3f})")

# 3. First-mover net sentiment N=5
col3 = "first_mover_net_sent_5"
if col3 in all_sample:
    axes[0,2].hist(all_sample[col3].dropna(), bins=30, edgecolor="black", alpha=0.7, color="green")
    axes[0,2].set_title("First-Mover Net Sentiment N=5"); axes[0,2].axvline(0, color="red", ls="--")

# 4. Crowding Gini
if "crowding_gini" in all_sample:
    axes[1,0].hist(all_sample["crowding_gini"].dropna(), bins=30, edgecolor="black", alpha=0.7, color="purple")
    axes[1,0].set_title("Crowding Gini (0=equal, 1=monopoly)")

# 5. Fresh blood ratio H=21
if "fresh_blood_ratio_21" in all_sample:
    axes[1,1].hist(all_sample["fresh_blood_ratio_21"].dropna(), bins=30, edgecolor="black", alpha=0.7, color="teal")
    axes[1,1].set_title("Fresh Blood Ratio H=21")

# 6. Specialist ratio
if "specialist_ratio" in all_sample:
    axes[1,2].hist(all_sample["specialist_ratio"].dropna(), bins=30, edgecolor="black", alpha=0.7, color="brown")
    axes[1,2].set_title("Specialist Ratio (fraction posting ?3 symbols)")

plt.tight_layout(); plt.show()

## 8. Feature Statistics & Validation Checks

In [ ]:
feature_cols = [c for c in all_sample.columns if c not in ("symbol", "date")]
print(f"Total feature columns: {len(feature_cols)}")
print("\nSummary statistics (selected features):")
display(all_sample[feature_cols[:12]].describe().round(4))

# Validation checks
print("\nValidation checks:")
ratio_cols = [c for c in feature_cols if "ratio" in c or "dominance" in c or "gini" in c]
for c in ratio_cols:
    if c in all_sample:
        mn, mx = all_sample[c].min(), all_sample[c].max()
        ok = (mn >= -0.01) and (mx <= 1.01)
        print(f"  {c:<40} min={mn:.3f}  max={mx:.3f}  {'OK' if ok else 'CHECK'}")

# Flip counts should be non-negative integers
for H in [5, 21]:
    for pfx in ("bear_to_bull_flips", "bull_to_bear_flips"):
        c = f"{pfx}_{H}"
        if c in all_sample:
            assert all_sample[c].min() >= 0, f"{c} has negative values"
print("\nAll non-negativity checks passed.")
print(f"\nNull value counts (top 10):")
print(all_sample[feature_cols].isnull().sum().sort_values(ascending=False).head(10))

## 9. Process All Years

In [ ]:
all_features = []
processing_stats = []

# Persistent carry state across years (sequential, chronological)
carry_state   = None  # initialised on first call
prev_daily_dir = None  # for flip/conviction carry-forward

print(f"{'='*60}")
print("Processing all years (chronological order)...")
print(f"{'='*60}\n")

for file in tqdm(files, desc="Processing years"):
    try:
        year = file.split("_")[-1].replace(".csv", "")
        df_year = pd.read_csv(INPUT_FOLDER / file)
        df_year["date"]       = pd.to_datetime(df_year["date"])
        df_year["created_at"] = pd.to_datetime(df_year["created_at"])

        # ?? flip features ??????????????????????????????????????????????
        # FIX (integration review): both flips and conviction must receive the
        # PRIOR years' directions.  The original loop reassigned prev_daily_dir
        # to THIS year's directions before calling conviction, so conviction
        # concatenated the current year with itself -- duplicating every
        # (user, symbol, date) row in the 250-td window, inflating streaks and
        # double-counting users.  See features_05_conviction_fix.md.
        prior_daily_dir = prev_daily_dir
        flip_yr, cur_daily_dir = calc_flip_features(
            df_year, horizons=HORIZONS, prev_daily_dir=prior_daily_dir
        )

        # ?? conviction ?????????????????????????????????????????????????
        conv_yr = calc_conviction_features(
            df_year, k_values=K_VALUES, prev_daily_dir=prior_daily_dir
        )

        # Prepare carry for NEXT year: this year's directions, trimmed to the
        # last 250 trading days (moved here from above so the reassignment
        # can no longer leak into this year's conviction call).
        prev_daily_dir = cur_daily_dir
        if prev_daily_dir is not None and len(prev_daily_dir) > 0:
            cutoff_td = td_idx.get(df_year["date"].max(), 0) - 250
            cutoff_date = td_sorted[max(0, cutoff_td)]
            prev_daily_dir = prev_daily_dir[
                prev_daily_dir["date"] >= cutoff_date
            ].copy()

        # ?? first-mover ????????????????????????????????????????????????
        fm_yr = calc_first_mover_features(df_year, n_values=N_VALUES)

        # ?? cohort (carry state persists) ?????????????????????????????
        cohort_yr, carry_state = calc_cohort_features(
            df_year, carry=carry_state, horizons=[21, 63]
        )

        # ?? merge all feature families ????????????????????????????????
        yr_feats = (
            flip_yr
            .merge(conv_yr,   on=["symbol","date"], how="outer")
            .merge(fm_yr,     on=["symbol","date"], how="outer")
            .merge(cohort_yr, on=["symbol","date"], how="outer")
            .sort_values(["symbol","date"]).reset_index(drop=True)
        )
        all_features.append(yr_feats)
        processing_stats.append({
            "year": year, "input_rows": len(df_year),
            "feature_rows": len(yr_feats),
            "unique_symbols": yr_feats["symbol"].nunique(),
        })

    except Exception as e:
        print(f"  ERROR in {file}: {e}")
        import traceback; traceback.print_exc()

features_all = pd.concat(all_features, ignore_index=True)
print(f"\nDone!  Total feature rows: {len(features_all):,}")
print(f"Unique symbols: {features_all['symbol'].nunique():,}")
print(f"Date range: {features_all['date'].min()} to {features_all['date'].max()}")
display(pd.DataFrame(processing_stats))

## 10. Final Data Inspection

In [ ]:
print(f"Final dataset: {features_all.shape}")
print(f"\nAll columns ({len(features_all.columns)}):")
print(sorted(features_all.columns.tolist()))
print(f"\nMemory usage: {features_all.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nNull counts (non-zero only):")
nulls = features_all.isnull().sum()
print(nulls[nulls > 0])
print("\nSummary statistics:")
display(features_all.describe().round(4))

## 11. Save to Pickle

In [ ]:
print(f"Saving to: {OUTPUT_FILE}")
features_all.to_pickle(OUTPUT_FILE)
print(f"Saved.")

verify = pd.read_pickle(OUTPUT_FILE)
assert verify.shape == features_all.shape
assert list(verify.columns) == list(features_all.columns)
file_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"Verified. File size: {file_mb:.1f} MB")

## Summary

**Features Computed**

| # | Feature | Column Prefix | Notes |
|---|---------|---------------|-------|
| 5 | Bear-to-Bull Flip Count | `bear_to_bull_flips_H` | H ? {5,21,63,250} td |
| 6 | Bull-to-Bear Flip Count | `bull_to_bear_flips_H` | H ? {5,21,63,250} td |
| 7 | Net Flip Count | `net_flip_count_H` | |
| 8 | Bear-to-Bull Flip Ratio | `bear_to_bull_ratio_H` | ? tagged users today |
| 9 | Bull-to-Bear Flip Ratio | `bull_to_bear_ratio_H` | |
| 10 | Conviction Index | `conviction_index_K` | K ? {3,5,10} posts |
| 16 | First-Mover Sentiment | `first_mover_net_sent_N` | N ? {5,10,50} |
| 17-18 | Fresh Blood Count / Ratio | `fresh_blood_count_H`, `fresh_blood_ratio_H` | H ? {21,63} |
| 19 | Re-entry Volume | `reentry_volume_H` | |
| 20 | Whale Dominance | `whale_dominance` | top 1% global cumulative posters |
| 21 | Minnow Dominance | `minnow_dominance` | < 5 lifetime posts |
| 22 | Night-Owl Ratio | `night_owl_ratio` | > 80% after-hours (expanding) |
| 23 | Day-Trader Ratio | `day_trader_ratio` | < 20% after-hours |
| 24 | Retention Rate | `retention_rate_H` | |
| 25 | Crowding Index (Gini) | `crowding_gini` | |
| 26 | Specialist Ratio | `specialist_ratio` | ? 3 unique symbols historically |

**Output:** `features_05_sentiment_dynamics_cohorts.pkl`

**Design notes**
- Flip/conviction carry previous year's daily directions in `prev_daily_dir` (trimmed to 250-td window).
- Cohort carry state persists across all 14 years via `carry_state` dict updated after each year.
- Night-owl / day-trader and specialist ratio use expanding (all-history) windows, not rolling.
  Rolling variants are a straightforward extension once a message-body-level dataset is available.
- Feature 27 (Sector Expert Ratio) requires an external sector-permno mapping not present in the
  current pipeline; deferred.
